# NB9 — Integrity check: train/test leakage

Step 4 trained on `Iqra_Extra_IS26` (real errors) and we report F1 on
`QuranMB.v2`. **If those share clips/speakers/utterances, the 0.667 is inflated.**
This notebook checks, before we trust or extend the number:
  1. **ID overlap** — same clip IDs in both?
  2. **Audio-array overlap** — identical waveforms (same recording, different ID)?
  3. **Sentence/phoneme overlap** — same verses (expected & fine) vs same
     *spoken-error* sequences (would be leakage).

Honest expectation: the challenge treats Extra_IS26 as TRAIN and QuranMB as TEST,
so they SHOULD be disjoint. We verify rather than assume.

In [ ]:
!pip install -q "datasets==2.21.0"
import os, numpy as np, hashlib
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN:", e)
from datasets import load_dataset, Audio
SR = 16_000

In [ ]:
extra = load_dataset("IqraEval/Iqra_Extra_IS26", split="train")
qmb   = load_dataset("IqraEval/QuranMB.v2", split="test")
print("Extra_IS26:", extra.column_names, len(extra))
print("QuranMB.v2:", qmb.column_names, len(qmb))

In [ ]:
# ---- 1) ID overlap ----------------------------------------------------------
extra_ids = set(str(x) for x in extra["id"])
qmb_ids   = set(str(x) for x in qmb["ID"])
id_overlap = extra_ids & qmb_ids
print(f"Extra IDs: {len(extra_ids)} | QuranMB IDs: {len(qmb_ids)}")
print(f"ID overlap: {len(id_overlap)}")
if id_overlap:
    print("  examples:", list(id_overlap)[:10])
    print("  >>> LEAKAGE via shared IDs.")
else:
    print("  OK: no shared IDs.")

In [ ]:
# ---- 2) Audio-array overlap (same recording under a different ID) -----------
# Hash a downsampled fingerprint of each waveform; compare sets. Identical
# fingerprints = the same audio appears in both train and test = hard leakage.
def fp(arr):
    a = np.asarray(arr, dtype=np.float32)
    if len(a) > 4000:
        idx = np.linspace(0, len(a)-1, 4000).astype(int)
        a = a[idx]
    a = np.round(a, 3)
    return hashlib.md5(a.tobytes()).hexdigest()

extra = extra.cast_column("audio", Audio(sampling_rate=SR))
qmb   = qmb.cast_column("audio", Audio(sampling_rate=SR))
extra_fp = set(fp(extra[i]["audio"]["array"]) for i in range(len(extra)))
qmb_fp   = set(fp(qmb[i]["audio"]["array"])   for i in range(len(qmb)))
audio_overlap = extra_fp & qmb_fp
print(f"audio fingerprint overlap: {len(audio_overlap)}")
print("  >>> LEAKAGE: identical audio in train & test." if audio_overlap
      else "  OK: no identical waveforms shared.")

In [ ]:
# ---- 3) Spoken-error sequence overlap --------------------------------------
# Same SENTENCE/canonical across both is EXPECTED (both recite Quran) -> not
# leakage. The concern is the same SPOKEN-ERROR (phoneme_mis) pattern. We can't
# compare to QuranMB's hidden GT here, but we report how many Extra phoneme_mis
# strings are IDENTICAL to its own phoneme_ref (i.e. clips with no error), which
# tells us how 'error-rich' the training signal really is.
import collections
mis = [str(s) for s in extra["phoneme_mis"]]
ref = [str(s) for s in extra["phoneme_ref"]]
n_with_error = sum(1 for m, r in zip(mis, ref) if m.split() != r.split())
print(f"Extra_IS26: {n_with_error}/{len(mis)} clips actually contain an error "
      f"(phoneme_mis != phoneme_ref).")
print(f"distinct sentences in Extra: {len(set(str(s) for s in extra['sentence']))}")

### Verdict
- **ID overlap = 0 AND audio overlap = 0** -> the 0.667 is CLEAN; it's a fair,
  defensible number you can take to the client.
- **Any audio/ID overlap > 0** -> the model saw test clips in training; the 0.667
  is inflated and we must retrain excluding the overlap, then re-score.
- Shared *sentences* alone are NOT leakage (everyone recites the same Quran); only
  shared audio/IDs (or shared spoken-error recordings) are.